# Laboratório de dois corpos — completo, escrito por você

**Pedro Henrique G. Modesto** · Mestrado IFSC/USP · orientação Lucas Madeira

Este notebook contém **toda a simulação de dois corpos** do projeto, num arquivo só.
Nenhuma linha de código está escrita. Cada célula tem comentários dizendo exatamente
o que fazer — inclusive os gráficos. Você escreve, roda, e confere com o alvo.

---

## Como funciona

- Toda célula de código começa com `# ---- ESCREVA AQUI ----` e uma lista numerada.
- Quando houver mais de um jeito de fazer, eu comento **as alternativas e o porquê**.
- Todo exercício tem um **alvo**: um número ou uma figura. Se bater, está certo.
- Não importe nada de `src/`. Aqui é do zero, com numpy puro.

## A convenção (a mesma do `src/`, para transferir direto)

`hbar = massa_reduzida = 1`, comprimentos em fm. A equação que resolvemos é

$$u''(r) = 2\,V(r)\,u(r)$$

e a resposta é o **comprimento de espalhamento**, que chamo de `a`.

## Nomenclatura (nomes completos, sem abreviação)

| nome no código | o que é |
|---|---|
| `comprimento_espalhamento` (`a`) | onde a reta externa da função de onda cruza o zero |
| `alcance_efetivo` (`r0`) | quanto a função de onda verdadeira difere dessa reta |
| `profundidade` (`v`) | profundidade adimensional do poço |
| `inverso_alcance` (`mu`) | inverso do alcance do potencial; `R = 1/mu` |
| `funcao_onda_radial` (`u`) | a solução de `u'' = 2Vu` |

---
# Parte 0 · As ferramentas

Só duas bibliotecas, o tempo inteiro.

## Ex 1 · Ligar as ferramentas

**Alvo:** imprimir a versão do numpy.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. importe numpy com o apelido np
# 2. importe matplotlib.pyplot com o apelido plt
# 3. imprima np.__version__
#
# ALTERNATIVA: existe `%matplotlib inline`. No Jupyter moderno ele ja e o padrao,
# entao so use se os graficos nao aparecerem.




## Ex 2 · A grade radial

**Duas formas de criar, e elas servem a propósitos diferentes:**

| forma | quando usar |
|---|---|
| `np.linspace(inicio, fim, n)` | quando você quer **n pontos** exatos |
| `np.arange(n) * dr` | quando você quer um **passo `dr`** exato |

No integrador vamos precisar do **passo** explícito, então a segunda.

**Alvo:** `r` com 801 pontos de 0 a 8,0, e `r[1] - r[0]` valendo `0.01`.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. dr = 0.01
# 2. n = 801
# 3. r = np.arange(n) * dr
# 4. imprima r[0], r[-1] e r[1]-r[0]




---
# Parte 1 · Os quatro potenciais

Todos têm dois parâmetros: `profundidade` (`v`) e `inverso_alcance` (`mu`).

## Ex 3 · Poço esférico

$$V(r) = -v\,\mu^2 \quad (r < R), \qquad 0 \quad (r \ge R), \qquad R = 1/\mu$$

**Três formas de escrever o "se" em numpy:**

| forma | comentário |
|---|---|
| `np.where(cond, sim, nao)` | **a que eu recomendo** — clara e vetorizada |
| `V = np.zeros_like(r); V[r<=R] = -v*mu**2` | funciona, duas linhas, mais explícita |
| `-v*mu**2 * (r <= R)` | booleano vira 0/1; compacto mas obscuro |

**Alvo:** `V_poco(np.array([0.5, 1.5]), 2.0, 1.0)` deve dar `[-2., 0.]`.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina V_poco(r, v, mu)
# 2. R = 1.0/mu
# 3. devolva np.where(r <= R, -v*mu**2, 0.0)
# 4. teste com np.array([0.5, 1.5]), v=2.0, mu=1.0
#
# CUIDADO: use r <= R*(1+1e-12), nao r <= R. Ponto flutuante nao acerta
# a igualdade exata, e a borda do poco tem que ficar DENTRO.




## Ex 4 · Os outros três

$$V_{\rm gauss} = -v\mu^{2}e^{-r^{2}\mu^{2}}
\qquad
V_{\rm mPT} = \frac{-v\mu^{2}}{\cosh^{2}(\mu r)}
\qquad
V_{\rm LJ} = \tfrac12\!\left(\frac{C_{12}}{r^{12}} - \frac{C_{6}}{r^{6}}\right)$$

**O `1/2` do Lennard-Jones é o ACHADO Nº 1 deste laboratório.** Ele **não** está
na Eq. (121) publicada do artigo. Sem ele a Tabela 4 não fecha. Com ele, fecha
em 0,1%.

**Alvo:** `V_gauss(0.0, 2.0, 1.0)` e `V_mpt(0.0, 2.0, 1.0)` devem dar `-2.0`.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina V_gauss(r, v, mu)   -> use np.exp
# 2. defina V_mpt(r, v, mu)     -> use np.cosh
# 3. defina V_lj(r, C12, C6)    -> nao esqueca o fator 1/2
# 4. teste os dois primeiros em r = 0.0
#
# CUIDADO no Lennard-Jones: ele DIVERGE em r -> 0. Se voce avaliar em r=0
# recebe inf ou nan. Na pratica o codigo comeca a grade num r_min onde o
# potencial ja e enorme mas finito.




## Ex 5 · Ver os quatro — **primeiro gráfico, você desenha**

**Alvo:** quatro curvas, todas negativas perto da origem e subindo para zero.
O poço tem um **degrau vertical**; os outros são lisos. Guarde essa diferença:
ela volta a te morder no Ex 17.

**Formas de montar a figura:**

| forma | quando |
|---|---|
| `plt.plot(...)` direto | rápido, um painel só |
| `fig, ax = plt.subplots()` | **recomendo** — você controla o eixo, e precisa disso para vários painéis |
| `fig, (ax1, ax2) = plt.subplots(1, 2)` | dois painéis lado a lado (Ex 17) |

In [ ]:
# ---- ESCREVA AQUI ----
# 1. rr = np.linspace(0.01, 5, 500)
# 2. fig, ax = plt.subplots(figsize=(7,4))
# 3. ax.plot(rr, V_poco(rr, 1.0, 1.0), label='poco esferico')
#    ax.plot(rr, V_gauss(rr, 1.0, 1.0), label='gaussiano')
#    ax.plot(rr, V_mpt(rr, 1.0, 1.0), label='Poschl-Teller modificado')
# 4. ax.axhline(0, color='k', lw=0.5)
# 5. ax.set_xlabel('r (fm)') ; ax.set_ylabel('V(r)')
# 6. ax.legend() ; ax.grid(alpha=0.3) ; plt.show()




---
# Parte 2 · Resolver a equação

Da diferença central de segunda ordem,
$$u''(r_i) \simeq \frac{u_{i+1} - 2u_i + u_{i-1}}{(\Delta r)^2} = 2V_i u_i$$

isolando:
$$\boxed{u_{i+1} = 2u_i - u_{i-1} + 2(\Delta r)^2 V_i u_i}$$

Condições de contorno: `u[0] = 0` (regularidade na origem) e `u[1] = 1`
(a normalização é livre — a equação é **linear**).

## Ex 6 · A grade alinhada

Parece burocracia e **não é**. O poço tem uma descontinuidade em `r = R`. Se essa
borda cair no meio de um passo, você perde **uma ordem inteira** de convergência.

**Alvo:** `grade(R, dr)` devolve `(r, N, dr_efetivo)` com `r[N] == R` exatamente,
`len(r) == N+2` (um ponto **além** de `R`, que a derivada vai precisar) e
`dr_efetivo <= dr`.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina grade(R, dr, r_min=0.0)
# 2. N = int(np.ceil((R - r_min)/dr - 1e-12))
# 3. dr_ef = (R - r_min)/N
# 4. r = r_min + dr_ef*np.arange(N+2)
# 5. devolva r, N, dr_ef
#
# POR QUE o -1e-12: sem ele, quando (R-r_min)/dr da exatamente um inteiro,
# o ceil pode subir um por erro de arredondamento e voce ganha um ponto a mais.




## Ex 7 · O integrador de diferença central

**Duas formas, e uma delas não funciona:**

| forma | veredito |
|---|---|
| laço `for` explícito | **a certa.** É uma recorrência: cada passo depende do anterior |
| vetorizar com numpy | **impossível.** Recorrência não vetoriza — cada `u[i+1]` precisa de `u[i]` |
| `scipy.integrate.solve_ivp` | funciona, mas é caça-mosca-com-canhão e esconde o método |

**Alvo:** `u[0]==0`, `u[1]==1`, e com `V=0` a solução é a reta exata `u[i] = i`.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina integrar(V, dr)
#      a) u = np.zeros(len(V))
#      b) u[1] = 1.0
#      c) laco for i in range(1, len(V)-1):
#             u[i+1] = 2*u[i] - u[i-1] + 2*dr**2 * V[i] * u[i]
#      d) devolva u
# 2. teste com V = np.zeros(50), dr=0.01 -> u[10] deve dar 10.0




## Ex 8 · O Numerov

Sobe de ordem 2 para ordem 4 quase de graça. Escrevendo `u'' = -xi(r)u` com
`xi = -2V`, e `h2 = (Δr)²/12`:

$$u_{i+1} = \frac{2u_i(1 - 5h_2\xi_i) - u_{i-1}(1 + h_2\xi_{i-1})}{1 + h_2\xi_{i+1}}$$

**Alvo:** com o poço `v=0.5, mu=1.0, dr=1e-3`, os dois métodos devem concordar
nas primeiras casas.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina numerov(V, dr)
#      a) xi = -2.0*V
#      b) h2 = dr*dr/12.0
#      c) u = np.zeros(len(V)) ; u[1] = 1.0
#      d) laco i de 1 ate len(V)-2:
#           u[i+1] = (2*u[i]*(1 - 5*h2*xi[i]) - u[i-1]*(1 + h2*xi[i-1])) / (1 + h2*xi[i+1])
#      e) devolva u
#
# CUIDADO: o denominador (1 + h2*xi[i+1]) pode zerar se o potencial for muito
# fundo e o passo grande. Se der divisao por zero, diminua dr.




## Ex 9 · **Gráfico:** o potencial e a solução

Dois painéis empilhados, compartilhando o eixo x. Em cima o potencial, embaixo
a função de onda.

**Alvo:** ver a onda **encurvar** dentro do poço e virar **reta** fora dele.
Essa é a figura que explica o assunto inteiro.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. escolha o gaussiano do deuteron: v=1.9102, mu=0.6754
# 2. r, N, dre = grade(9.0, 1e-3)
# 3. V = V_gauss(r, v, mu) ; u = numerov(V, dre)
# 4. fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8,7), sharex=True,
#                                   gridspec_kw={'height_ratios':[1,2]})
# 5. ax1.plot(r, V) ; ax1.set_ylabel('V(r)')
# 6. ax2.plot(r, u) ; ax2.set_xlabel('r (fm)') ; ax2.set_ylabel('u(r)')
# 7. ax1.axhline(0, color='k', lw=0.5) ; ax2.axhline(0, color='k', lw=0.5)
#
# O sharex=True e o que alinha os dois eixos x. Sem ele os paineis
# ficam com escalas diferentes e a comparacao visual quebra.




---
# Parte 3 · O comprimento de espalhamento

Fora do alcance `V = 0`, logo `u'' = 0`: a solução **é uma reta**. O comprimento
de espalhamento é onde ela cruza o zero.

$$u(r) \propto r - a$$

## Ex 10 · Duas formas de extrair, e por que uma é melhor

| forma | como | problema |
|---|---|---|
| **régua esticada** | pega dois pontos do fim, acha a reta, vê onde cruza zero | você tem que **escolher** os dois pontos. Com cauda longa, escolhe errado |
| **derivada logarítmica** | casa `u'/u` com a da reta, na borda `R` | **não depende da normalização.** É por isso que se usa |

A segunda, que é a Eq. (110) do artigo:

$$\boxed{a = R - \frac{2\,\Delta r\; u_N}{u_{N+1} - u_{N-1}}}$$

**Alvo:** poço `v=0.5, mu=1.0` deve dar `a = -0.557408`, que é exatamente o valor
da fórmula fechada.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina extrair_a(r, u, N, dr):
#      devolva r[N] - 2*dr*u[N]/(u[N+1] - u[N-1])
#
# 2. teste: R = 1.0 ; r, N, dre = grade(R, 1e-3)
#           u = numerov(V_poco(r, 0.5, 1.0), dre)
#           imprima extrair_a(r, u, N, dre)
#
# 3. (opcional) escreva tambem a versao "regua esticada" e compare.
#    Ela usa: inclinacao = (u[-1]-u[-100])/(r[-1]-r[-100]) e depois
#    a = r[-100] - u[-100]/inclinacao




## Ex 11 · A fórmula fechada do poço

$$a = R\left[1 - \frac{\tan x}{x}\right], \qquad x = \sqrt{2v}$$

**Alvo:** o limiar do primeiro estado ligado em `v = π²/8 = 1.2337`.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina a_poco_exato(v, R=1.0):
#      x = np.sqrt(2*v) ; devolva R*(1 - np.tan(x)/x)
# 2. imprima a_poco_exato(0.5, 1.0)   -> deve dar -0.557408
# 3. imprima np.pi**2/8                -> o limiar




## Ex 12 · **Gráfico:** numérico contra analítico

**Alvo:** as bolinhas do numérico **em cima** da linha do analítico, e a
divergência em `v = π²/8`.

**O obstáculo:** o polo. Se um ponto cair perto de `v = π²/8`, o valor vai para
`1e15` e o matplotlib escala o eixo y para acomodá-lo — o resto da curva vira
uma linha reta grudada no zero.

**Três formas de resolver:**

| forma | comentário |
|---|---|
| `ax.set_ylim(-12, 12)` | **a mais simples.** Corta a vista, mantém os dados |
| trocar valores grandes por `np.nan` | o matplotlib não desenha `nan` — deixa o buraco |
| plotar `1/a` em vez de `a` | **a mais física.** O polo vira uma raiz limpa |

In [ ]:
# ---- ESCREVA AQUI ----
# 1. vs = np.linspace(0.1, 3.0, 300)          -> para a linha analitica
#    vs_pontos = np.linspace(0.1, 3.0, 25)    -> para as bolinhas numericas
# 2. calcule a_poco_exato para vs
# 3. para cada v de vs_pontos: grade + V_poco + numerov + extrair_a
# 4. fig, ax = plt.subplots(figsize=(8,4.5))
# 5. ax.plot(vs, analitico, label='formula fechada')
#    ax.plot(vs_pontos, numerico, 'o', mfc='none', label='nosso codigo')
# 6. ax.axvline(np.pi**2/8, ls='--', color='k', label='limiar')
# 7. ax.set_ylim(-12, 12) ; ax.axhline(0, color='k', lw=0.5)
# 8. ax.set_xlabel('profundidade v') ; ax.set_ylabel('a (fm)') ; ax.legend()




---
# Parte 4 · O alcance efetivo

$$r_0 = 2\int_0^R \left[g_0^2(r) - u_0^2(r)\right]dr, \qquad g_0(r) = 1 - \frac{r}{a}$$

`g_0` é a reta assintótica **estendida para dentro** (a mentira), `u_0` é a função
verdadeira. A integral da diferença é o quanto o potencial de fato ocupa espaço.

## Ex 13 · Normalizar e montar o integrando

A normalização: multiplique `u` por `C` tal que `C·u(R) = g_0(R) = 1 - R/a`.

**Alvo:** um gráfico com `g_0²` (tracejado) e `u_0²` (cheio), com a área entre as
duas sombreada. Essa área vezes 2 é o alcance efetivo.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. resolva o gaussiano do deuteron: v=1.9102, mu=0.6754, R=8.0, dr=1e-3
# 2. a = extrair_a(...)
# 3. C = (1 - r[N]/a) / u[N]
# 4. un = C*u
# 5. g0 = 1 - r/a
# 6. integrando = g0**2 - un**2
# 7. fig, ax = plt.subplots(figsize=(8,4.5))
#    ax.plot(r, g0**2, '--', label='g0^2  (mundo ideal, alcance zero)')
#    ax.plot(r, un**2, label='u0^2  (mundo real)')
#    ax.fill_between(r, un**2, g0**2, alpha=0.25, label='area x 2 = r0')
#    ax.set_xlim(0, 6) ; ax.legend()




## Ex 14 · Trapézio e Simpson, na mão

**Por que escrever as duas?** Se duas quadraturas independentes derem o mesmo
número, você confia. Se discordarem, a grade está grossa. É um teste embutido.

- **Trapézio:** `Δr[½f₀ + f₁ + ... + f_{n-2} + ½f_{n-1}]`
- **Simpson:** `(Δr/3)[f₀ + 4f₁ + 2f₂ + 4f₃ + ... + f_{n-1}]`, com `n` ímpar

**Alvo:** `a ≈ 5.400` e `r0 ≈ 1.699`. O artigo publica 5,40 e 1,70.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina trapezio(f, dr):
#      devolva dr*(np.sum(f) - 0.5*f[0] - 0.5*f[-1])
#
# 2. defina simpson(f, dr):
#      pesos = np.ones(len(f)) ; pesos[1:-1:2] = 4 ; pesos[2:-1:2] = 2
#      devolva dr/3 * np.sum(pesos*f)
#      (se len(f) for PAR, corte o ultimo ponto antes: f = f[:-1])
#
# 3. r0_trap = 2*trapezio(integrando[:N+1], dre)
#    r0_simp = 2*simpson(integrando[:N+1], dre)
# 4. imprima a, r0_trap, r0_simp
#
# ALTERNATIVA: np.trapezoid e scipy.integrate.simpson fazem isso prontos.
# Use DEPOIS, para conferir que a sua versao bate.




---
# Parte 5 · Comparando os métodos — **o ACHADO Nº 2**

O Numerov é de ordem 4 e a diferença central é de ordem 2. Então o Numerov
sempre ganha, certo?

**Não.** E medir isso é o segundo achado deste laboratório.

## Ex 15 · Erro contra passo, em log-log

**Alvo — e é o ponto do exercício:**

| potencial | diferença central | Numerov |
|---|---|---|
| mPT (**liso**) | ordem 2 · erro `3.6e-07` | **já no piso, `2.8e-08`** |
| poço (**degrau**) | **ordem 2 · erro `3.5e-07`** | ordem 1 · erro `3.4e-04` |

*(erros com `dr = 1e-3`, contra a fórmula exata)*

No liso o Numerov ganha fácil. **No poço ele perde por mil vezes**, e a
inclinação da reta cai de 2 para 1.

**A razão:** a ordem alta do Numerov **pressupõe** potencial suave (classe C4).
Na borda do poço essa hipótese morre, e com ela a vantagem.

**Como ler o gráfico:** a **inclinação** da reta em log-log **é** a ordem do
método. Você não vai ler isso num livro — vai medir.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. drs = np.array([8e-3, 4e-3, 2e-3, 1e-3, 5e-4])
# 2. para cada dr e cada metodo, calcule |a_num/a_exato - 1|
#    - para o poco use a_poco_exato como referencia
#    - para o mPT nao ha formula simples aqui: use o proprio Numerov com
#      dr=1e-4 como referencia (e um truque legitimo, mas ANOTE que e isso)
# 3. fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11,4))
# 4. ax.loglog(drs, erros, 'o-', label=metodo)
# 5. meça a inclinacao: np.polyfit(np.log(drs), np.log(erros), 1)[0]
#    e imprima. Ela E a ordem do metodo.
#
# POR QUE loglog e nao plot: uma lei de potencia erro ~ dr^p vira uma RETA
# de inclinacao p no log-log. Em escala linear voce nao ve nada.




---
# Parte 6 · Universalidade

Quatro potenciais com formas completamente diferentes, ajustados para terem o
**mesmo** comprimento de espalhamento. Fora do alcance, as quatro soluções têm
que virar a **mesma reta**.

Se acontecer, você provou com as suas mãos que a baixa energia o formato do
potencial não importa — só dois números importam.

## Ex 16 · As curvas colapsando

Parâmetros do caso nêutron-nêutron (Tabela 3 do artigo), todos com
`a ≈ -18.5` fm:

| potencial | `v` | `mu` |
|---|---|---|
| Pöschl-Teller modificado | 0.9071 | 0.7991 |
| gaussiano | 1.2121 | 0.5672 |
| poço esférico | 1.1096 | 0.3918 |

**Alvo:** as três curvas separadas perto da origem, e **coladas** na mesma reta
`1 - r/(-18.5)` depois de `r ≈ 4` fm.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. para cada um dos tres: resolva com R=10, dr=1e-3, e normalize (como no Ex 13)
# 2. fig, ax = plt.subplots(figsize=(8,5))
# 3. plote as tres curvas normalizadas ate r=8, com cores diferentes
# 4. plote por cima, tracejada e grossa, a reta 1 - r/(-18.5)
# 5. imprima o `a` de cada um — os tres batem na primeira casa decimal
#
# DICA de leitura: o que importa nao e as curvas serem iguais (nao sao,
# perto da origem). E elas convergirem para a MESMA reta la fora.




---
# Parte 7 · Estados ligados, e o **ACHADO Nº 3**

Dois números viram uma energia:

$$E_{\rm zr} = -\frac{\hbar^2}{2m_r a^2}
\qquad\qquad
\frac{1}{a} = \kappa - \frac{r_0\kappa^2}{2},\;\; E_{\rm fr} = -\frac{\hbar^2\kappa^2}{2m_r}$$

`zr` = alcance zero (só `a`). `fr` = alcance finito (usa `a` **e** `r0`).

## Ex 17 · O dêuteron — quanto vale o alcance efetivo

Valores medidos: `a = 5.4112` fm, `r0 = 1.7436` fm, e `ħ²/(2m_r) = 41.47` MeV·fm².

**Alvo:**

| | energia | erro |
|---|---|---|
| alcance zero | −1,416 MeV | **36%** |
| alcance finito | −2,223 MeV | 0,05% |
| experimento | −2,224 MeV | — |

**O `r0` vale 0,8 MeV no dêuteron.** Parece detalhe e não é — e é exatamente o
tema do Madeira (2024), que é o fio que liga este laboratório à sua dissertação.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. h2_2mr = 41.47   # MeV fm^2
# 2. a_exp, r0_exp = 5.4112, 1.7436
# 3. E_zr = -h2_2mr/a_exp**2
# 4. kappa = (1 - np.sqrt(1 - 2*r0_exp/a_exp))/r0_exp
#    E_fr  = -h2_2mr*kappa**2
# 5. imprima os dois e compare com -2.224 MeV
#
# CUIDADO: a raiz (1 - 2*r0/a) fica NEGATIVA se r0 > a/2. Ai a formula nao
# tem raiz real e o estado nao e ligado do jeito que voce supos.




## Ex 18 · O dímero de hélio no fio da navalha

**O ACHADO Nº 3.** Mesmo átomo, dois potenciais:

| potencial | `a` | liga? |
|---|---|---|
| Lennard-Jones de de Boer | **−178 Å** | **não** (0 nós) |
| Aziz HFD-B | **+88,4 Å** | **sim** (1 nó, `E = −1,69 mK`) |

Sinal oposto de `a`. Quando `|a| ≫ alcance`, a resposta é **hipersensível** à
forma do potencial — e a universalidade não te protege disso, porque ela só
diz o que acontece **dado** o `a`, não qual `a` você tem.

**Alvo:** contar os nós de `u` e ver 0 num caso e 1 no outro.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina contar_nos(u):
#      s = np.sign(u[np.abs(u) > 0])
#      devolva int(np.count_nonzero(s[1:]*s[:-1] < 0))
#
# 2. use o Lennard-Jones com C12 = 262279.0, C6 = 940.586 (unidades de Angstrom)
#    grade de r_min = 0.415 ate R = 990, dr pequeno
# 3. conte os nos e extraia o `a`
#
# CUIDADO: com o caroco repulsivo do Lennard-Jones, u CRESCE exponencialmente
# e estoura o float. Solucao: dentro do laco, se abs(u[i+1]) > 1e250,
# divida o array inteiro por 1e250. A equacao e linear, entao pode.
#
# ATENCAO: esse mesmo reescalonamento causou um BUG REAL neste laboratorio.
# Ver Parte 9.




---
# Parte 8 · Incerteza e portões de validade

**Um número sem barra de erro não é resultado.** E um número dentro da barra de
erro mas **fora do regime de validade** da teoria com que você compara é pior:
parece certo e está errado.

## Ex 19 · Extrapolação de Richardson

Se o método é de ordem `p` e você tem o resultado em dois passos `h` e `h/2`:

$$f_{\rm extrapolado} = f_{h/2} + \frac{f_{h/2} - f_h}{2^p - 1}$$

E a **incerteza honesta** é `|f_extrapolado - f_{h/2}|`.

**Alvo:** `a = 5.4002699 ± 0.0000010 fm` para o gaussiano do dêuteron.

**E o bônus:** com três passos você pode **medir** a ordem em vez de supor:
$$p_{\rm medida} = \frac{\ln|(f_1-f_2)/(f_2-f_3)|}{\ln(h_1/h_2)}$$
Se ela discordar da esperada, a hipótese do método quebrou.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. passos = [4e-3, 2e-3, 1e-3]
# 2. calcule o `a` do gaussiano do deuteron em cada passo
# 3. ordem_medida = log(|(v0-v1)/(v1-v2)|) / log(passos[0]/passos[1])
# 4. extrapolado = v2 + (v2-v1)/(2**2 - 1)
# 5. incerteza = abs(extrapolado - v2)
# 6. imprima "a = {extrapolado} +- {incerteza}"  e a ordem medida
#
# FACA O MESMO com o Numerov no POCO e veja a ordem medida dar ~1 em vez
# de ~4. E o Achado numero 2 aparecendo como numero, nao como grafico.




## Ex 20 · O portão de validade — **uma descoberta que ninguém tinha automatizado**

Toda teoria universal (alcance zero) exige `|a| ≫ r0`. Se `|a|/r0 < 10`, você
está fora do regime e **não pode comparar** com teoria de alcance zero.

Quando eu apontei esse portão para a literatura, apareceu isto:

| sistema | `\|a\|/r0` | universal? |
|---|---|---|
| dêuteron (3S1) | **3,10** | **não** |
| par n-p singleto | 8,57 | não |
| dímero de ⁴He | 11,30 | sim |
| ³⁹K com `a = 1000 a₀` | **7,35** | **não** |

**O exemplo canônico de universalidade em livro-texto — o dêuteron — não está no
regime universal.** Não é erro de ninguém. Fica invisível até você automatizar.

**Alvo:** reproduzir essa tabela.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina portao_universalidade(a, r0, limiar=10.0):
#      razao = abs(a)/abs(r0)
#      devolva razao, razao > limiar
# 2. rode para os quatro sistemas da tabela acima
#    (deuteron 5.4112/1.7436 ; singleto -23.74/2.77 ;
#     helio 90.4/8.0 ; 39K 1000*0.0529/136*0.0529)
# 3. imprima uma tabela com a razao e o veredito




---
# Parte 9 · Onde a pesquisa está — o que descobrimos nos últimos dias

Esta parte **não tem exercício**. É o registro do que o laboratório produziu,
para o notebook ficar completo em si mesmo.

## Os três achados de dois corpos

**1 · Fator 2 na Eq. (121) do artigo.** As constantes de Lennard-Jones da
Tabela 4 só reproduzem `(a, r0)` com `V = ħ²/(2mᵣ)·[C₁₂/r¹² − C₆/r⁶]`. Você
usou isso no Ex 4.

**2 · Numerov perde a ordem alta em potencial descontínuo.** Na borda do poço
ele degrada de ordem 4 para ordem 1, e chega a errar **mil vezes mais** que a
diferença central. Você mediu isso no Ex 15 e no Ex 19.

**3 · O dímero de hélio no fio da navalha.** Lennard-Jones não liga
(`a = −178 Å`), Aziz liga (`a = +88,4 Å`). Ex 18.

## Três bugs numéricos silenciosos — todos deram número errado sem erro nenhum

**Underflow apaga os nós.** O reescalonamento contra overflow divide o array
inteiro por `1e250`. Quando dispara várias vezes, a parte **inicial** da solução
vira zero exato — 57.133 zeros de 60.000 pontos. Os nós somem.
**Conserto: contar os nós DURANTE a integração, não no fim.**

**Overflow ao detectar o nó.** `u[i+1]*u[i] < 0` estoura quando ambos são
`~1e250`. **Conserto: comparar sinais em vez de multiplicar.**

**Borda do poço desalinhada da grade.** Custa uma ordem inteira de convergência.
É o que a `grade()` do Ex 6 conserta.

## E a fronteira, em três corpos

O laboratório passou para três corpos e produziu quatro resultados
independentes, todos ancorados, sobre a **descrição por uma "dimensão efetiva"**
do crossover 3D → 2D:

| # | resultado | natureza |
|---|---|---|
| 1 | o espectro confinado **não é geométrico** (razões diferem 63%) | numérico |
| 2 | o termo desprezado satura em **71%** do termo mantido | geométrico |
| 3 | a energia de um canal erra por **√3 = 73,2%** | **exato** |
| 4 | a fidelidade máxima cai como **6,75/anisotropia** | **exato** |

O nº 4 é o mais forte: não fala de erro percentual, fala da descrição **deixar
de ser uma descrição**. Em anisotropia 100 — valor experimental modesto — a
melhor descrição possível de um canal tem **6% de sobreposição** com a função de
onda verdadeira.

**Ressalva permanente:** tudo isso é o caso **sem interação**, o mais amigável
possível para a teoria criticada. Com interação ressonante só pode piorar —
**mas isso ainda precisa ser mostrado, não afirmado.**

Detalhes, derivações e as autópsias dos resultados que morreram: `HISTORICO.md`.

---
# Parte 10 · Empacotar

## Ex 21 · Uma função que faz tudo

Você tem todas as peças. Agora junte.

**Alvo:** `calcular(V_gauss, (1.9102, 0.6754), R=8.0)` devolve
`a = 5.400`, `r0 = 1.699`.

Quando isso funcionar, você reescreveu `src/potenciais.py`, `src/solvers.py`,
`src/espalhamento.py` e `src/analitico.py` — com as suas mãos, sem ter olhado.

In [ ]:
# ---- ESCREVA AQUI ----
# 1. defina calcular(Vfunc, params, R, dr=1e-3, metodo='numerov')
#      a) r, N, dre = grade(R, dr)
#      b) V = Vfunc(r, *params)
#      c) u = numerov(V, dre) se metodo=='numerov' senao integrar(V, dre)
#      d) a = extrair_a(r, u, N, dre)
#      e) normalize, monte o integrando, calcule r0 com simpson
#      f) devolva a, r0
#
# 2. teste com o gaussiano do deuteron
# 3. teste com os tres casos nn e confira contra -18.5 / 2.70




---
# Fechamento

| você escreveu | no repositório é |
|---|---|
| `V_poco`, `V_gauss`, `V_mpt`, `V_lj` | `src/potenciais.py` |
| `grade`, `integrar`, `numerov` | `src/solvers.py` |
| `extrair_a`, `trapezio`, `simpson`, `calcular` | `src/espalhamento.py` |
| `a_poco_exato` | `src/analitico.py` |
| Richardson e o portão de validade | `src/incerteza.py` |

## Onde procurar os números

```bash
python referencias/literatura.py            # indice
python referencias/literatura.py gauss      # tudo sobre o gaussiano
python referencias/literatura.py deuteron
python referencias/literatura.py D1         # a divergencia do fator 2
```

## O que vem depois

Três corpos: `src/trimero.py`, com 11 testes de âncora. A torre de Efimov, a
razão 22,694, e o que a armadilha faz com ela.

E a fronteira: o multicanal, que é onde a crítica vira construção.